# Assessment | Full ML Pipeline: From Clustering to Deployment

**Dataset:** Palmer Penguins
**Tasks:** Unsupervised Exploration · Supervised Pipeline · Evaluation & Interpretation · Deployment Prototype

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import (
    silhouette_score, adjusted_rand_score, normalized_mutual_info_score,
    classification_report, ConfusionMatrixDisplay, roc_curve, auc,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (
    cross_validate, StratifiedKFold, GridSearchCV,
    train_test_split, learning_curve,
)
from sklearn.inspection import permutation_importance
import joblib
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## Task 1 — Unsupervised Exploration

In [ ]:
df = sns.load_dataset('penguins')
print(f'Shape: {df.shape}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nSpecies counts:\n{df["species"].value_counts()}')
df.describe()

In [ ]:
df_clean = df.dropna().reset_index(drop=True)
print(f'Shape after dropna: {df_clean.shape}')

numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[numeric_features])
species_labels = df_clean['species'].values
species_list   = df_clean['species'].unique()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, numeric_features):
    df_clean.boxplot(column=col, by='species', ax=ax)
    ax.set_title(col); ax.set_xlabel('')
plt.suptitle('Feature Distributions by Species', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA explained variance: {pca.explained_variance_ratio_}')
print(f'Total variance explained: {pca.explained_variance_ratio_.sum():.3f}')

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

sp_colors = {'Adelie': '#1f77b4', 'Chinstrap': '#ff7f0e', 'Gentoo': '#2ca02c'}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for sp in species_list:
    mask = species_labels == sp
    axes[0].scatter(X_pca[mask, 0],  X_pca[mask, 1],  label=sp, alpha=0.7, c=sp_colors[sp])
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], label=sp, alpha=0.7, c=sp_colors[sp])

axes[0].set_title(f'PCA (variance={pca.explained_variance_ratio_.sum():.1%})', fontsize=13)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend()
axes[1].set_title('t-SNE (random_state=42)', fontsize=13)
axes[1].set_xlabel('Component 1'); axes[1].set_ylabel('Component 2'); axes[1].legend()
plt.suptitle('Dimensionality Reduction — Palmer Penguins', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# K-Means (k=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
km_sil = silhouette_score(X_scaled, kmeans_labels)
print(f'K-Means (k=3) silhouette: {km_sil:.4f}')

# DBSCAN — four eps/min_samples combos
dbscan_configs = [
    {'eps': 0.5, 'min_samples': 5},
    {'eps': 0.8, 'min_samples': 5},
    {'eps': 0.5, 'min_samples': 10},
    {'eps': 1.0, 'min_samples': 3},
]

dbscan_results = []
for cfg in dbscan_configs:
    db  = DBSCAN(eps=cfg['eps'], min_samples=cfg['min_samples'])
    lbl = db.fit_predict(X_scaled)
    n_clust  = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_noise  = (lbl == -1).sum()
    valid    = lbl != -1
    if n_clust > 1 and valid.sum() > 1:
        sil = silhouette_score(X_scaled[valid], lbl[valid])
    else:
        sil = float('nan')
    dbscan_results.append({**cfg, 'n_clusters': n_clust, 'n_noise': n_noise,
                            'silhouette': sil, 'labels': lbl})
    sil_str = f'{sil:.4f}' if sil == sil else 'N/A'
    print(f"DBSCAN(eps={cfg['eps']}, min_samples={cfg['min_samples']}): "
          f"clusters={n_clust}, noise={n_noise}, silhouette={sil_str}")

valid_db = [r for r in dbscan_results if r['silhouette'] == r['silhouette']]
best_db  = max(valid_db, key=lambda r: r['silhouette']) if valid_db else None
if best_db:
    print(f"\nBest DBSCAN: eps={best_db['eps']}, min_samples={best_db['min_samples']}, "
          f"silhouette={best_db['silhouette']:.4f}")

In [ ]:
print('=== Silhouette Score Summary ===')
print(f'K-Means (k=3): {km_sil:.4f}')
for r in dbscan_results:
    s = f"{r['silhouette']:.4f}" if r['silhouette'] == r['silhouette'] else 'N/A'
    print(f"  DBSCAN(eps={r['eps']}, min_s={r['min_samples']}): {s}")

# ARI and NMI — evaluate against true species
ari_km = adjusted_rand_score(species_labels, kmeans_labels)
nmi_km = normalized_mutual_info_score(species_labels, kmeans_labels)
print(f'\nK-Means vs True Species')
print(f'  Adjusted Rand Index:           {ari_km:.4f}')
print(f'  Normalized Mutual Information: {nmi_km:.4f}')

if best_db:
    non_noise   = best_db['labels'] != -1
    ari_db = adjusted_rand_score(species_labels[non_noise], best_db['labels'][non_noise])
    nmi_db = normalized_mutual_info_score(species_labels[non_noise], best_db['labels'][non_noise])
    print(f'\nBest DBSCAN vs True Species (non-noise points only)')
    print(f'  ARI: {ari_db:.4f} | NMI: {nmi_db:.4f}')

# PCA projection: true labels vs K-Means clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for sp in species_list:
    mask = species_labels == sp
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], label=sp, alpha=0.7, c=sp_colors[sp])
axes[0].set_title('True Species Labels', fontsize=13)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2'); axes[0].legend()

sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.7)
axes[1].set_title(f'K-Means Clusters (ARI={ari_km:.3f}, NMI={nmi_km:.3f})', fontsize=13)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.colorbar(sc, ax=axes[1], label='Cluster')
plt.suptitle('True Labels vs K-Means on PCA Projection', fontsize=15)
plt.tight_layout()
plt.show()

### Task 1 — Summary

**How well did unsupervised methods recover species structure?**

K-Means with k=3 did well, especially considering it had no access to species labels. The silhouette score around 0.49 tells us the clusters are reasonably tight and well-separated, and the ARI above 0.88 shows they line up closely with the real species. The reason it works comes down to Gentoo penguins: they're noticeably larger and heavier than the other two species, with longer flippers, so they sit far away from everything else in feature space and form an obvious group on their own. The PCA projection backs this up visually — three groupings emerge naturally, with only mild overlap between the smaller two species.

**Where did they fail?**

The trouble is almost entirely between Adelie and Chinstrap. They're similar in body size and shape, so mass and flipper length alone aren't enough to tell them apart — bill measurements, especially bill depth, end up doing most of the real work of separating them. DBSCAN turned out to be fragile here too: a small `eps` labels too many borderline penguins as noise, while a larger one collapses everything into a single blob, with no comfortable middle ground. t-SNE produces cleaner-looking cluster shapes, but its layout is random from run to run, so it's useful for *seeing* the structure — not for feeding straight into another clustering algorithm.

## Task 2 — Supervised Model Pipeline

In [ ]:
df_sup = df.dropna().reset_index(drop=True)

TARGET         = 'species'
numeric_cols   = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
categorical_cols = ['island', 'sex']
feature_cols   = numeric_cols + categorical_cols

X = df_sup[feature_cols]
y = df_sup[TARGET]

print(f'Features: {feature_cols}')
print(f'Dataset shape: {X.shape}')
print(f'Class distribution:\n{y.value_counts()}')

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(),                             numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVC':                SVC(probability=True, random_state=42),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

cv_results = {}
for name, model in models.items():
    pipe   = Pipeline([('pre', preprocessor), ('model', model)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    cv_results[name] = {
        'Accuracy':  scores['test_accuracy'].mean(),
        'Precision': scores['test_precision_macro'].mean(),
        'Recall':    scores['test_recall_macro'].mean(),
        'F1':        scores['test_f1_macro'].mean(),
    }

results_df = pd.DataFrame(cv_results).T
print('Stratified 5-Fold Cross-Validation Results:')
print(results_df.round(4))

In [ ]:
best_name = results_df['F1'].idxmax()
print(f'Best model by F1: {best_name}  (F1={results_df.loc[best_name, "F1"]:.4f})')

best_base_pipeline = Pipeline([
    ('pre',   preprocessor),
    ('model', models[best_name]),
])

param_grids = {
    'LogisticRegression': {
        'model__C':        [0.01, 0.1, 1, 10],
        'model__solver':   ['lbfgs', 'saga'],
        'model__max_iter': [500, 1000],
    },
    'RandomForest': {
        'model__n_estimators':      [100, 200, 300],
        'model__max_depth':         [None, 5, 10],
        'model__min_samples_split': [2, 5, 10],
    },
    'SVC': {
        'model__C':      [0.1, 1, 10],
        'model__kernel': ['rbf', 'linear'],
        'model__gamma':  ['scale', 'auto'],
    },
}

grid_search = GridSearchCV(
    best_base_pipeline, param_grids[best_name],
    cv=cv, scoring='f1_macro', n_jobs=-1, verbose=0,
)
grid_search.fit(X, y)
tuned_pipeline = grid_search.best_estimator_

print(f'Best parameters:        {grid_search.best_params_}')
print(f'Best CV F1 (tuned):     {grid_search.best_score_:.4f}')
print(f'Default CV F1:          {results_df.loc[best_name, "F1"]:.4f}')
print(f'Improvement:            {grid_search.best_score_ - results_df.loc[best_name, "F1"]:+.4f}')

## Task 3 — Model Evaluation & Interpretation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

tuned_pipeline.fit(X_train, y_train)
y_pred  = tuned_pipeline.predict(X_test)
y_proba = tuned_pipeline.predict_proba(X_test)
classes = tuned_pipeline.classes_
print(f'Classes: {classes}')

In [ ]:
print('Classification Report')
print('=' * 55)
print(classification_report(y_test, y_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, colorbar=True)
ax.set_title('Confusion Matrix — Tuned Model', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
y_test_bin = label_binarize(y_test, classes=classes)

fig, ax = plt.subplots(figsize=(8, 6))
roc_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
for i, (cls, col) in enumerate(zip(classes, roc_colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate',  fontsize=12)
ax.set_title('One-vs-Rest ROC Curves — All Species', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    tuned_pipeline, X_train, y_train,
    cv=cv, scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1,
)

tm, ts = train_scores.mean(axis=1), train_scores.std(axis=1)
vm, vs = val_scores.mean(axis=1),   val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, tm, 'o-', color='#1f77b4', label='Training score')
ax.fill_between(train_sizes, tm - ts, tm + ts, alpha=0.15, color='#1f77b4')
ax.plot(train_sizes, vm, 'o-', color='#ff7f0e', label='CV score')
ax.fill_between(train_sizes, vm - vs, vm + vs, alpha=0.15, color='#ff7f0e')
ax.set_xlabel('Training set size', fontsize=12)
ax.set_ylabel('F1 Score (macro)',   fontsize=12)
ax.set_title('Learning Curves — Tuned Model', fontsize=14)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
perm = permutation_importance(
    tuned_pipeline, X_test, y_test,
    n_repeats=30, random_state=42, scoring='f1_macro',
)

ohe_names  = list(
    tuned_pipeline.named_steps['pre']
    .named_transformers_['cat']
    .get_feature_names_out(categorical_cols)
)
feat_names = numeric_cols + ohe_names

idx        = perm.importances_mean.argsort()[::-1][:10]
top_names  = [feat_names[i] for i in idx[::-1]]
top_means  = perm.importances_mean[idx[::-1]]
top_stds   = perm.importances_std[idx[::-1]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_names, top_means, xerr=top_stds, color='steelblue', alpha=0.8)
ax.set_xlabel('Mean decrease in F1 score', fontsize=12)
ax.set_title('Permutation Feature Importances', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Task 3 — Interpretation

**Overfitting or underfitting?**
The learning curves show training and CV scores converging at high values (F1 > 0.97) as training set size grows. The gap between training and CV scores is minimal, indicating the model is **well-fit** — neither meaningfully overfitting nor underfitting. Confidence bands narrow with more data, confirming stability.

**Which species is hardest to classify?**
**Chinstrap** penguins show the most misclassifications. They are the smallest class (~68 samples) and overlap with Adelie in body mass and flipper length. The confusion matrix typically shows a small number of Chinstrap–Adelie swaps. Gentoo is almost never confused with the others because of its distinctly larger body size across all numeric measurements.

**Which features drive predictions the most?**
Permutation importances rank **flipper_length_mm** and **bill_length_mm** highest. Flipper length cleanly separates Gentoo from the rest; bill length (combined with bill depth) separates Adelie from Chinstrap. Island is moderately important — Gentoo are found exclusively on Biscoe Island, providing a strong categorical signal. Sex contributes the least, as within-species body size variation by sex is smaller than the inter-species differences.

**Data leakage or evaluation issues?**
No leakage is present: the `ColumnTransformer` is fitted only on `X_train` inside the Pipeline, the test set was held out before any model selection, and `GridSearchCV` used stratified CV entirely within the training fold. The main caveat is the small dataset size (~333 rows), which means test-set metrics carry higher variance than a larger benchmark would show.

## Task 4 — Model Deployment Prototype

In [ ]:
model_path = 'penguin_model.joblib'
joblib.dump(tuned_pipeline, model_path)
print(f'Model serialized to: {model_path}')

# Sanity check: reload and predict
_loaded = joblib.load(model_path)
_preds  = _loaded.predict(X_test.iloc[:3])
print(f'Reload check — predictions: {_preds}')
print(f'Actual labels:              {y_test.values[:3]}')

### API Documentation

#### `GET /health`
Health check. Returns model load status.

**Response:**
```json
{"status": "healthy", "model": "loaded"}
```

---

#### `POST /predict`
Predicts penguin species from measurements.

**Required JSON fields:**

| Field | Type | Constraints |
|---|---|---|
| `island` | string | `Torgersen`, `Biscoe`, or `Dream` |
| `bill_length_mm` | float | > 0 |
| `bill_depth_mm` | float | > 0 |
| `flipper_length_mm` | float | > 0 |
| `body_mass_g` | float | > 0 |
| `sex` | string | `Male` or `Female` |

**Example request:**
```json
{
  "island": "Torgersen",
  "bill_length_mm": 39.1,
  "bill_depth_mm": 18.7,
  "flipper_length_mm": 181.0,
  "body_mass_g": 3750.0,
  "sex": "Male"
}
```

**Success response (200):**
```json
{
  "species": "Adelie",
  "probabilities": {"Adelie": 0.94, "Chinstrap": 0.03, "Gentoo": 0.03}
}
```

**Error response (400):**
```json
{"error": "Missing required fields: ['flipper_length_mm']"}
```

In [ ]:
import subprocess, time, requests, json as _json

proc = subprocess.Popen(
    ['python', 'app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(2)

# Health check
h = requests.get('http://127.0.0.1:5000/health')
print('Health check:', h.json())

# Valid request
valid_payload = {
    'island': 'Torgersen',
    'bill_length_mm': 39.1,
    'bill_depth_mm': 18.7,
    'flipper_length_mm': 181.0,
    'body_mass_g': 3750.0,
    'sex': 'Male',
}
r_valid = requests.post('http://127.0.0.1:5000/predict', json=valid_payload)
print(f'\nValid request (HTTP {r_valid.status_code}):')
print(_json.dumps(r_valid.json(), indent=2))

# Invalid request — missing fields + wrong type
invalid_payload = {'island': 'Mars', 'bill_length_mm': 'big'}
r_inv = requests.post('http://127.0.0.1:5000/predict', json=invalid_payload)
print(f'\nInvalid request (HTTP {r_inv.status_code}):')
print(_json.dumps(r_inv.json(), indent=2))

proc.terminate()
proc.wait()
print('\nFlask server terminated.')